# RG3 F1: 기계/온도 흔들림 동시 확인

기존rank75 RF/ET4모델 평균 점수에 쿠션과 배럴2/5의 정상 조건부 순위 동시 특이도(min/max)를 곱한다. 상단순위/양단순위/세변수 모두상단 확인을 사전 정의. 이 조건은 물리적 가스 원인 증명이 아니다. 기존점수/확인점수 단독·두점수평균 선택과 임계값은 내부OOF F1로만 결정한다.

고정RG3 RH620/가스19/정상601·같은7개구간. 캐시 모델과 순위의 학습ref가 평가구간을 제외한지 assert. 평가 라벨로 확인조건 방향/임계값을 바꾸지 않는다. 알려진 자료 개발용 중첩검증이며 새 독립검증 아님. 전문가는 기본4모델 평균 앙상블에 정해진 확인 조건을 적용한 것.


In [1]:
from pathlib import Path
import numpy as np,pandas as pd
from sklearn.metrics import confusion_matrix,precision_recall_curve,average_precision_score
root=Path.cwd()
if not (root/'data').exists():root=root.parent
t=root/'reports/tables';z=np.load(t/'rg3_F1_boost_fold_cache.npz',allow_pickle=False)
y=z['labels'];groups=z['groups'];times=z['timestamps'];blocks=np.unique(groups)
assert len(y)==620 and y.sum()==19
cache={}
for key in z.files:
 if key.startswith('ref_'):
  n=key.split('_')[1];ref=tuple(z[key].astype(int));rank=z['features_'+n+'_1'];p=z['scores_'+n][:,1]
  r=rank[:,[6,8,9]]
  upper=np.minimum(r[:,1],np.maximum(r[:,0],r[:,2]))
  bilateral=2*np.abs(r-.5)
  agreement=np.minimum(bilateral[:,1],np.maximum(bilateral[:,0],bilateral[:,2]))
  all_upper=np.min(r,axis=1)
  cache[ref]=np.column_stack([p,p*upper,p*agreement,p*all_upper])
def get(ref,q):
 assert set(groups[ref]).isdisjoint(groups[q])
 return cache[tuple(ref)][q]
def stats(yy,p):
 tn,fp,fn,tp=confusion_matrix(yy,p,labels=[0,1]).ravel()
 return dict(TP=int(tp),FP=int(fp),FN=int(fn),TN=int(tn),F1=2*tp/(2*tp+fp+fn) if 2*tp+fp+fn else 0,recall=tp/(tp+fn) if tp+fn else 0)
def cutfit(yy,score):
 p,r,c=precision_recall_curve(yy,score);f=np.divide(2*p[:-1]*r[:-1],p[:-1]+r[:-1],out=np.zeros_like(c),where=p[:-1]+r[:-1]>0)
 cut=float(c[np.lexsort((c,r[:-1],f))[-1]]);return cut,stats(yy,score>=cut)
mixes=[np.eye(4)[i] for i in range(4)]+[(np.eye(4)[i]+np.eye(4)[j])/2 for i in range(4) for j in range(i+1,4)]
records=[];choices=[]
for b in blocks:
 q=np.flatnonzero(groups==b);ref=np.flatnonzero(groups!=b);out=get(ref,q);inner=np.full((len(ref),4),np.nan)
 for ib in np.unique(groups[ref]):
  mask=groups[ref]==ib;inner[mask]=get(ref[~mask],ref[mask])
 candidates=[]
 for w in mixes:
  cut,result=cutfit(y[ref],inner@w);candidates.append((result['F1'],result['recall'],-result['FP'],w,cut,out@w))
 best=max(candidates,key=lambda a:a[:3])
 named=[('inner_F1_selected_noise_confirmation',best[5],best[4])]
 for i,name in enumerate(['rank75_reference','cushion_thermal_upper_confirm','cushion_thermal_bilateral_confirm','all_three_upper_confirm']):
  cut,_=cutfit(y[ref],inner[:,i]);named.append((name+'_inner_F1_cut',out[:,i],cut))
 for name,score,cut in named:
  for j,i in enumerate(q):records.append(dict(model=name,TimeStamp=times[i],block=int(b),actual=int(y[i]),prediction=int(score[j]>=cut),score=float(score[j]),threshold=cut))
 choices.append(dict(block=int(b),weights=','.join(map(str,best[3])),threshold=best[4],inner_F1=best[0]))
p=pd.DataFrame(records);m=pd.DataFrame([dict(model=name,AP=average_precision_score(g.actual,g.score),**stats(g.actual,g.prediction)) for name,g in p.groupby('model')])
for obj,name in [(p,'predictions'),(m,'metrics'),(pd.DataFrame(choices),'inner_choices')]:obj.to_csv(t/f'rg3_noise_confirmation_{name}.csv',index=False,encoding='utf-8-sig')
print(m.to_string(index=False))


                                         model       AP  TP  FP  FN  TN       F1   recall
          all_three_upper_confirm_inner_F1_cut 0.038595  11 231   8 370 0.084291 0.578947
cushion_thermal_bilateral_confirm_inner_F1_cut 0.055806   3  20  16 581 0.142857 0.157895
    cushion_thermal_upper_confirm_inner_F1_cut 0.056527  10 169   9 432 0.101010 0.526316
          inner_F1_selected_noise_confirmation 0.082488   5  35  14 566 0.169492 0.263158
                 rank75_reference_inner_F1_cut 0.076694   5  35  14 566 0.169492 0.263158
